# Stochastic Calculus, Step by Step · Lesson 24: Delta hedging in practice

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The recipe, on one path · 3. The hedging error · 4. Rebalancing more often · 5. The drift doesn't matter · 6. The wrong volatility · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 24,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. The recipe, on one path

Sell the call for its Black-Scholes price. Buy $\Delta = N(d_+)$ shares, financing the difference in cash at rate $r$. At each rebalance, the cash earns interest, and you trade to the new delta. At expiry, pay what the option owes. Twelve monthly steps on one path:

In [ ]:
delta_of = lambda s, tau, sig=sigma: Nv(d_plus(s, tau, sig))


def hedge(n_reb, n_paths, gen, real_mu=mu, real_vol=sigma, price_vol=sigma, kind="call", detail=False):
    # Sell the option at price_vol, delta-hedge at price_vol, while the share really moves with real_mu and real_vol.
    dt = T / n_reb
    f = call if kind == "call" else put
    s = np.full(n_paths, S0, float)
    pos = delta_of(s, T, price_vol) if kind == "call" else delta_of(s, T, price_vol) - 1
    cash = float(f(S0, T, price_vol)) - pos * s
    rows = [{"t": 0.0, "S": S0, "delta": float(pos[0]), "cash": float(cash[0]), "option": float(f(S0, T, price_vol))}] if detail else None
    for i in range(1, n_reb + 1):
        s = s * np.exp((real_mu - 0.5 * real_vol**2) * dt + real_vol * math.sqrt(dt) * gen.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        tau = T - i * dt
        if i < n_reb:
            new = delta_of(s, tau, price_vol) if kind == "call" else delta_of(s, tau, price_vol) - 1
            cash = cash - (new - pos) * s
            pos = new
            if detail:
                rows.append({"t": i * dt, "S": float(s[0]), "delta": float(pos[0]), "cash": float(cash[0]), "option": float(f(s[0], tau, price_vol))})
    pay = np.maximum(s - K, 0) if kind == "call" else np.maximum(K - s, 0)
    err = pos * s + cash - pay
    return (err, rows, s, pay, pos, cash) if detail else err


rng = np.random.default_rng(seed)
one = hedge(12, 1, rng, detail=True)
out["path"] = {"rows": one[1], "S_T": float(one[2][0]), "payoff": float(one[3][0]), "shares_end": float(one[4][0] * one[2][0]),
               "cash_end": float(one[5][0]), "error": float(one[0][0]), "price": float(call(S0, T))}
pd.DataFrame(one[1])

## 3. The hedging error

Between rebalances the hedge is frozen, and the option's value curves away from it. Over one step, by Taylor and the Black-Scholes equation (Lesson 19), the error a short-option hedger takes on is

$$\text{error on the step} \approx \tfrac12\Gamma S^2\Big(\sigma^2\Delta t - \big(\tfrac{\Delta S}{S}\big)^2\Big) .$$

Each step's error has mean about zero, so they add up to something with mean about zero: the size of the moves, not their direction, is what counts. Exact one-step errors against the formula:

In [ ]:
dt_w = 1 / 52
one_rows = []
v0, d0, g0 = float(call(S0, T)), float(delta_of(S0, T)), float(phi(d_plus(S0, T)) / (S0 * sigma))
for m in (0.0, 0.01, -0.01, 0.0277, -0.0277, 0.05, -0.05):
    s1 = S0 * (1 + m)
    exact = -(float(call(s1, T - dt_w)) - v0) + d0 * (s1 - S0) + r * (v0 - d0 * S0) * dt_w
    one_rows.append({"move_pct": m * 100, "exact": exact, "approx": 0.5 * g0 * S0**2 * (sigma**2 * dt_w - m**2)})
out["step"] = {"rows": one_rows, "gamma": g0, "be_pct": sigma * math.sqrt(dt_w) * 100}
pd.DataFrame(one_rows)

## 4. Rebalancing more often

Weekly, daily, more. The error's average stays near zero; its standard deviation halves each time the rebalances are multiplied by 4. So standard deviation times $\sqrt n$ should settle to a constant, and for an option near the money a rule of thumb says that constant is about $\sqrt{\pi/4}\,\sigma\mathcal V$, with $\mathcal V$ the vega.

In [ ]:
paths = 40_000
n_rows, hists = [], {}
for n_reb in (4, 12, 52, 252, 1000):
    e_ = hedge(n_reb, paths, rng)
    n_rows.append({"n": n_reb, "mean": float(e_.mean()), "se": float(e_.std(ddof=1) / math.sqrt(paths)), "sd": float(e_.std(ddof=1)),
                   "sd_pct": float(e_.std(ddof=1) / call(S0, T) * 100), "p05": float(np.quantile(e_, 0.05)), "scaled": float(e_.std(ddof=1) * math.sqrt(n_reb))})
    if n_reb in (12, 252):
        c_, ed = np.histogram(e_, bins=np.linspace(-8, 8, 81), density=True)
        hists[f"n{n_reb}"] = c_.tolist()
        hists["edges"] = ed.tolist()
out["reb"] = {"rows": n_rows, "hists": hists, "rule": math.sqrt(math.pi / 4) * sigma * float(S0 * phi(d_plus(S0, T)) * math.sqrt(T)), "ratio_252_1000": n_rows[3]["sd"] / n_rows[4]["sd"], "paths": paths,
              "fig_n": [x["n"] for x in n_rows], "fig_sd": [x["sd"] for x in n_rows]}
pd.DataFrame(n_rows)

## 5. The drift doesn't matter

The hedge was built with no $\mu$ in it. Change the real-world expected return, and the unhedged seller's result changes completely, while the hedger's average stays near zero. The hedger's spread does shift a little, because $\mu$ changes where the share spends its time, and so how much gamma the hedger carries.

In [ ]:
mu_rows = []
for m in (-0.1, 0.0, 0.08, 0.2, 0.4):
    g = np.random.default_rng(seed + 1)
    e_ = hedge(252, 20_000, g, real_mu=m)
    g = np.random.default_rng(seed + 2)
    s_ = S0 * np.exp((m - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * g.standard_normal(200_000))
    naked = float(call(S0, T)) * math.exp(r * T) - np.maximum(s_ - K, 0)
    mu_rows.append({"mu_pct": m * 100, "mean": float(e_.mean()), "sd": float(e_.std(ddof=1)), "naked_mean": float(naked.mean()), "naked_sd": float(naked.std())})
out["drift"] = {"rows": mu_rows}
pd.DataFrame(mu_rows)

## 6. The wrong volatility

The hedge cares about volatility. Sell and hedge at 20% when the share really moves with 25%, and you lose money on average, about the difference of the two Black-Scholes prices. If it really moves with 15%, you make about that.

In [ ]:
vol_rows = []
for v_true in (0.15, 0.18, 0.20, 0.22, 0.25, 0.30):
    g = np.random.default_rng(seed + 3)
    e_ = hedge(252, 20_000, g, real_vol=v_true)
    diff = float(call(S0, T, sigma) - call(S0, T, v_true))
    vol_rows.append({"true_pct": v_true * 100, "mean": float(e_.mean()), "fv_diff": diff * math.exp(r * T), "sd": float(e_.std(ddof=1)),
                     "vega_approx": float(S0 * phi(d_plus(S0, T)) * math.sqrt(T) * (sigma - v_true)) * math.exp(r * T)})
out["wrong"] = {"rows": vol_rows}
pd.DataFrame(vol_rows)

In [ ]:
# Chart data for the video (no random numbers)
rows_ = out["path"]["rows"]
out["path"]["fig"] = {"t": [x_["t"] for x_ in rows_], "delta": [x_["delta"] for x_ in rows_],
                      "t_S": [x_["t"] for x_ in rows_] + [T], "S": [x_["S"] for x_ in rows_] + [out["path"]["S_T"]],
                      "ticks": [{"x": m_ / 12, "label": f"{m_} months" if m_ else "start"} for m_ in (0, 3, 6, 9, 12)]}
e_ = np.array(out["reb"]["hists"]["edges"])
out["reb"]["hists"]["mid"] = ((e_[:-1] + e_[1:]) / 2).tolist()
out["reb"]["hists"]["ticks"] = [{"x": float(x_), "label": f"{x_:.0f}"} for x_ in (-8, -4, 0, 4, 8)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Using Taylor's theorem to second order and the Black-Scholes equation, derive the one-step error of a short-option hedger: $\tfrac12\Gamma S^2\big(\sigma^2\Delta t - (\Delta S/S)^2\big)$.

**Exercise 2.** Treat the one-step errors as independent, with $\Gamma S^2$ roughly constant. Using $\mathrm{Var}\big((\Delta S/S)^2\big) \approx 2\sigma^4\Delta t^2$, show that the standard deviation of the total error is proportional to $1/\sqrt n$. What should the ratio of the errors at $n = 252$ and $n = 1000$ be?

**Exercise 3.** With one week between rebalances, how large a move leaves the hedger exactly even? What about one day? Compute both with $\sigma = 20\%$, and explain why the answer has nothing to do with the option.

**Exercise 4.** The share really moves with volatility $\sigma_{\text{true}}$, and you sell and hedge at $\sigma$. Use the one-step error to find the expected profit per unit of time, and show that its present value is $\tfrac12(\sigma^2 - \sigma_{\text{true}}^2)\int_0^Te^{-rt}E[S_t^2\Gamma_t]\,dt$. Which way does the profit go?

**Exercise 5.** Explain from the one-step error why the expected return $\mu$ doesn't affect the hedger at leading order. Where does it enter at all? Why doesn't this help you pick shares?

**Exercise 6.** Use vega to approximate the mean profit from selling at $\sigma = 20\%$ when the true volatility is 25%: $\text{mean} \approx \mathcal V\cdot(\sigma - \sigma_{\text{true}})$. With $\mathcal V = 39.89$, what is it, and how does it compare with the exact difference of the two Black-Scholes prices?

**Exercise 7.** (Code) Hedge daily and record for each path the realised volatility $\sqrt{\sum(\Delta\ln S)^2/T}$ and the final error. Sort the paths into buckets by realised volatility. Is the error related? What is the correlation, and does the relationship change if the path's final price is near the strike or far from it?

**Exercise 8.** (Code) Delta-hedge a sold put (the delta is $N(d_+) - 1$) with $n = 12, 52, 252$. Show that the standard deviations are close to the call's, and explain why, using put-call parity.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")